# 04 · Team shape out of possession
Width, depth, compactness (convex hull area) and last-line height of each team's
outfield players, frame by frame, split by defensive phase type.

In [ ]:
import sys
from pathlib import Path

# Make the repo's src/ importable no matter where the notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src.loading import match_table, load_tracking, load_match_meta, load_phases, orient
from src.shape import team_shape

matches = match_table()
MATCH_ID = int(matches.match_id.iloc[0])
meta = load_match_meta(MATCH_ID)
trk = load_tracking(MATCH_ID, step=5)
phases = load_phases(MATCH_ID)
print(f"{meta['home_team']} v {meta['away_team']}: {trk.frame.nunique():,} frames")

In [ ]:
# Map each frame to its phase of play
def phase_for_frames(frames, phases):
    p = phases.sort_values("frame_start")
    idx = np.searchsorted(p.frame_start.values, frames, side="right") - 1
    idx = np.clip(idx, 0, len(p) - 1)
    inside = frames <= p.frame_end.values[idx]
    out = p.iloc[idx][["index", "team_in_possession_id", "team_out_of_possession_phase_type"]].reset_index(drop=True)
    out.loc[~inside, :] = None
    out["frame"] = frames
    return out

frames = np.sort(trk.frame.unique())
frame_phase = phase_for_frames(frames, phases)
frame_phase.team_out_of_possession_phase_type.value_counts(dropna=False)

In [ ]:
# Shape of each team while it is out of possession, oriented so it defends the goal at -x
shapes = []
for side in ["home", "away"]:
    team_id = meta[f"{side}_team_id"]
    o = orient(trk[trk.team_id == team_id], MATCH_ID, team_id)
    s = team_shape(o, team_id).merge(frame_phase, on="frame", how="left")
    s = s[s.team_in_possession_id.notna() & (s.team_in_possession_id != team_id)]
    shapes.append(s.assign(team=meta[f"{side}_team"]))
shapes = pd.concat(shapes, ignore_index=True)
print(f"{len(shapes):,} defending frames")
shapes.head()

In [ ]:
blocks = ["high_block", "medium_block", "low_block"]
cols = ["width", "depth", "area", "line_height", "share_detected"]
(shapes[shapes.team_out_of_possession_phase_type.isin(blocks)]
   .groupby(["team", "team_out_of_possession_phase_type"])[cols].median().round(1))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, c in zip(axes, ["width", "depth", "area", "line_height"]):
    for t in blocks:
        v = shapes.loc[shapes.team_out_of_possession_phase_type == t, c].dropna()
        ax.hist(v, bins=30, alpha=0.5, density=True, label=t)
    ax.set_title(c)
axes[0].legend(fontsize=8)
plt.tight_layout()

In [ ]:
# Compactness over the match for one team
team = shapes.team.iloc[0]
s = shapes[shapes.team == team].sort_values("frame")
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(s.frame / 600, s.area.rolling(20, min_periods=5).median(), lw=1)
ax.set_xlabel("frame / 600 (≈ minutes)"); ax.set_ylabel("hull area (m²), rolling median")
ax.set_title(f"{team}: compactness while defending")
plt.show()

**Caveat:** shape measures use extrapolated positions for off-camera players.
`share_detected` shows how much of each frame's shape is estimated; consider filtering on it.

In [ ]:
shapes.groupby(pd.cut(shapes.share_detected, [0, 0.25, 0.5, 0.75, 1.0], include_lowest=True),
               observed=True)[["area", "width", "depth"]].median().round(1)